# India Power Demand & Weather Analysis: Exploratory Data Analysis

This notebook presents the empirical investigation answering our three core business questions:
1. **How strongly does temperature associate with electricity demand across India?**
2. **How are solar and wind altering the daily net load curve ("duck curve")?**
3. **What is the frequency and distribution of extreme anomaly days and ramping events?**

All metrics and charts are computed directly from the local modeled SQLite database (`data/energy_marts.db` fallback). These are descriptive associations, not causal estimates or forecasts. Temperatures are an unweighted six-city mean; selected telemetry has gaps.


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root))
from src.utils.config import get_db_path
conn = sqlite3.connect(f'{get_db_path().as_uri()}?mode=ro', uri=True)
print("Connected to database successfully.")


## 1. Daily Demand Evolution & Peak Load Growth
**Question:** How has aggregate demand evolved across seasons, and when did India record its maximum load in this extract?


In [ ]:
df_daily = pd.read_sql("""
    SELECT date_ist, AVG(demand_mw) as avg_demand_mw, MAX(demand_mw) as peak_demand_mw
    FROM mart_net_load_profile
    GROUP BY date_ist
    ORDER BY date_ist
""", conn)
df_daily['date'] = pd.to_datetime(df_daily['date_ist'])
df_daily['rolling_7d'] = df_daily.set_index('date')['avg_demand_mw'].rolling('7D').mean().to_numpy()

peak_val = df_daily['peak_demand_mw'].max()
peak_row = df_daily.loc[df_daily['peak_demand_mw'].idxmax()]

plt.figure(figsize=(14, 5))
plt.plot(df_daily['date'], df_daily['avg_demand_mw'], alpha=0.4, label='Daily Average (MW)')
plt.plot(df_daily['date'], df_daily['rolling_7d'], color='#0072B2', lw=2, label='7-Day Moving Average')
plt.scatter([peak_row['date']], [peak_val], color='#D55E00', s=80, zorder=5, label=f"Observed peak: {int(peak_val):,} MW")
plt.title(f"India Maximum Observed Demand Reached {int(peak_val):,} MW on {peak_row['date_ist']}")
plt.ylabel("Demand (MW)")
plt.legend()
plt.show()


## 2. Temperature Sensitivity & Cooling Demand
**Question:** At what rate does India's electricity demand increase per degree Celsius above the 25°C threshold?


In [ ]:
df_temp = pd.read_sql("""
    SELECT avg_temperature_c, demand_mw, season
    FROM mart_temp_vs_demand
""", conn)

cooling = df_temp[df_temp['avg_temperature_c'] >= 25.0].dropna(subset=['avg_temperature_c', 'demand_mw'])
if len(cooling) < 3 or cooling['avg_temperature_c'].nunique() < 2:
    raise ValueError('At least three complete cooling observations and two temperatures are required.')
slope, intercept = np.polyfit(cooling['avg_temperature_c'], cooling['demand_mw'], 1)

plt.figure(figsize=(14, 6))
sns.scatterplot(data=df_temp, x='avg_temperature_c', y='demand_mw', hue='season', alpha=0.3, s=20)
x_vals = np.linspace(25, df_temp['avg_temperature_c'].max(), 50)
plt.plot(x_vals, slope * x_vals + intercept, color='black', lw=2.5, linestyle='--', label=f"Cooling Gradient: +{int(slope):,} MW/°C")
plt.axvline(25.0, color='gray', linestyle=':')
plt.title(f"Descriptive Cooling Association: +{int(slope):,} MW for Every 1°C Above 25°C")
plt.xlabel("Average Temperature (°C)")
plt.ylabel("Grid Demand (MW)")
plt.legend()
plt.show()


## 3. Solar Ingress and the "Duck Curve"
**Question:** How does midday solar generation reshape the net load curve that dispatchable generators must serve?


In [ ]:
df_net = pd.read_sql("""
    SELECT season, hour_ist, AVG(demand_mw) as gross_demand, AVG(solar_mw) as solar, AVG(net_load_mw) as net_load
    FROM mart_net_load_profile
    GROUP BY season, hour_ist
""", conn)

plt.figure(figsize=(14, 6))
for s in df_net['season'].unique():
    sub = df_net[df_net['season'] == s]
    plt.plot(sub['hour_ist'], sub['net_load'], label=f"Net Load: {s}", lw=2)

plt.title("Diurnal Net Load Profile by Season: Midday Solar Depression")
plt.xlabel("Hour of Day (IST)")
plt.ylabel("Net Load (MW)")
plt.legend()
plt.show()


## 4. Net Load Ramping Stress & Anomaly Days
**Question:** What are the steepest 1-hour ramping events in the available telemetry?


In [ ]:
df_ramps = pd.read_sql("""
    SELECT abs_ramp_1h_mw, ramp_1h_mw, timestamp_ist
    FROM mart_net_load_ramps
    ORDER BY abs_ramp_1h_mw DESC
    LIMIT 10
""", conn)
print("Top 10 Steepest Net Load Ramps:")
display(df_ramps)


In [ ]:
conn.close()